In [ ]:
import collections, difflib, json, os, re, time, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

OUTPUT_DIR = "baza_import"                               

CSV_PATH   = "../dane/fra_cleaned_final.csv"
EMB_DIR    = "wyniki_embeddingow"         
CLU_DIR    = "wyniki_klasteryzacji"       
OUTPUT_DIR = "baza_import"                   
CONFIG = dict(OUTPUT_DIR=OUTPUT_DIR)         
assert all(os.path.exists(p) for p in (CSV_PATH, os.path.join(EMB_DIR, "Z_final.npy"), os.path.join(CLU_DIR, "perfume_clusters.csv"))), "brak plikow wejsciowych"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH, sep=";", encoding="utf-8-sig")
Z = np.load(os.path.join(EMB_DIR, "Z_final.npy"))
index = pd.read_csv(os.path.join(EMB_DIR, "perfume_index.csv"), sep=";", encoding="utf-8-sig")
pcl = pd.read_csv(os.path.join(CLU_DIR, "perfume_clusters.csv"))
N = len(df)

id_z_url = df.url.str.extract(r"-(\d+)\.html$")[0].astype(float)
kontrole = {
    "liczba perfum: CSV = Z = indeks = klastry":              N == Z.shape[0] == len(index) == len(pcl),
    "url: CSV = indeks = klastry (ta sama kolejnosc)":        bool((df.url.values == index.url.values).all() and (df.url.values == pcl.url.values).all()),
    "perfume_id (Fragrantica): indeks = klastry":             bool((index.perfume_id.values == pcl.perfume_id.values).all()),
    "perfume_id = liczba na koncu adresu url":                bool((id_z_url.values == index.perfume_id.values).all()),
    "perfume_id unikalne":                                    bool(index.perfume_id.is_unique),
    "url unikalne":                                           bool(df.url.is_unique),
    "wektory: 512 wymiarow, dlugosc 1, bez NaN":              Z.shape[1] == 512 and bool(np.allclose(np.linalg.norm(Z, axis=1), 1, atol=1e-5)) and bool(np.isfinite(Z).all()),
}
for k, v in kontrole.items():
    print(f"{'OK ' if v else 'BLAD'}  {k}")
assert all(kontrole.values()), "zrodla sie rozjechaly - nie budujemy bazy"

tok = lambda seria: [t.strip() for x in seria.dropna() for t in x.split(",") if t.strip()]           

print("\nbraki (tylko kolumny z brakami):", df.isna().sum()[lambda s: s > 0].to_dict())
print("plec:", df.Gender.value_counts().to_dict(), "| rok:", int(df.Year.min()), "-", int(df.Year.max()),
      "| ocena:", df["Rating Value"].min(), "-", df["Rating Value"].max(), "| liczba ocen:", int(df["Rating Count"].min()), "-", int(df["Rating Count"].max()))

kraje = sorted(df.Country.unique())
print(f"\nKRAJE: {len(kraje)} wartosci; ze spacja na brzegu: {[k for k in kraje if k != k.strip()]}")
print("  podobne zapisy:", [(a, b) for i, a in enumerate(kraje) for b in kraje[i + 1:] if difflib.SequenceMatcher(None, a.strip().lower(), b.strip().lower()).ratio() > 0.85])
print("  marek z wiecej niz jednym krajem:", int((df.groupby("Brand").Country.nunique() > 1).sum()), "| marek:", df.Brand.nunique())

perf = tok(df.Perfumers)
cnt = collections.Counter(perf)
rodziny_zapisu = collections.defaultdict(set)
for k in cnt:
    rodziny_zapisu[re.sub(r"[^a-z]", "", k)].add(k)
print(f"\nPERFUMERZY: {len(cnt)} roznych wpisow (w tym 'unknown'), 'unknown' w {int((df.Perfumers.str.strip() == 'unknown').sum())} perfumach ({(df.Perfumers.str.strip() == 'unknown').mean():.0%}); max {int(df.Perfumers.str.count(',').max()) + 1} perfumerow na perfume")
print("  rozne zapisy tej samej osoby:", [sorted(v) for v in rodziny_zapisu.values() if len(v) > 1])
print("  wpisy z dziwnymi znakami:", [k for k in cnt if re.search(r"[\x00-\x1f\x7f-\x9f]|\?", k)])

nuty_w_warstwach = {w: tok(df[w]) for w in ["Top", "Middle", "Base"]}
wszystkie_nuty = collections.Counter(t for w in nuty_w_warstwach.values() for t in w)
powiazania = sum(len({t.strip() for t in x.split(",") if t.strip()}) for w in ["Top", "Middle", "Base"] for x in df[w])
ta_sama_w_dwoch = sum(1 for t, m, b in zip(df.Top, df.Middle, df.Base)
                      if len({*t.split(", "), *m.split(", "), *b.split(", ")}) < len(t.split(", ")) + len(m.split(", ")) + len(b.split(", ")))
print(f"\nNUTY: {len(wszystkie_nuty)} roznych, w >=100 perfumach: {sum(v >= 100 for v in wszystkie_nuty.values())}, w 1 perfumie: {sum(v == 1 for v in wszystkie_nuty.values())}; "
      f"powiazan perfuma-nuta-warstwa: {powiazania}; perfum z ta sama nuta w >1 warstwie: {ta_sama_w_dwoch}")
print("  nuty z nawiasami:", [k for k in wszystkie_nuty if re.search(r"[\(\[]", k)], "| nuty 'unknown':", wszystkie_nuty.get("unknown", 0))

akordy = pd.concat([df[f"mainaccord{i}"] for i in range(1, 6)]).dropna()
print(f"\nAKORDY: {akordy.nunique()} roznych; wpisow (perfuma, pozycja): {len(akordy)}; braki na pozycjach 1-5: {[int(df[f'mainaccord{i}'].isna().sum()) for i in range(1, 6)]}")
print("  perfum z tym samym akordem dwa razy:", int(sum(len(set(r.dropna())) < r.notna().sum() for _, r in df[[f"mainaccord{i}" for i in range(1, 6)]].iterrows())))

dup = df.duplicated(["Brand", "Perfume"], keep=False)
print(f"\nDUPLIKATY (ta sama marka i nazwa, rozne url): {int(df.duplicated(['Brand', 'Perfume']).sum())} nadmiarowych wpisow w {df[dup].groupby(['Brand', 'Perfume']).ngroups} grupach -> to osobne wpisy Fragrantiki, wiec zostaja")

OK   liczba perfum: CSV = Z = indeks = klastry
OK   url: CSV = indeks = klastry (ta sama kolejnosc)
OK   perfume_id (Fragrantica): indeks = klastry
OK   perfume_id = liczba na koncu adresu url
OK   perfume_id unikalne
OK   url unikalne
OK   wektory: 512 wymiarow, dlugosc 1, bez NaN

braki (tylko kolumny z brakami): {'Year': 2034, 'mainaccord2': 13, 'mainaccord3': 114, 'mainaccord4': 388, 'mainaccord5': 979}
plec: {'women': 11371, 'unisex': 7695, 'men': 4980} | rok: 1781 - 2024 | ocena: 2.41 - 4.93 | liczba ocen: 26 - 29858

KRAJE: 54 wartosci; ze spacja na brzegu: ['Arabia saudi ']
  podobne zapisy: [('Bahrain', 'Bahrein'), ('Iceland', 'Ireland')]
  marek z wiecej niz jednym krajem: 0 | marek: 1060

PERFUMERZY: 910 roznych wpisow (w tym 'unknown'), 'unknown' w 12310 perfumach (51%); max 2 perfumerow na perfume
  rozne zapisy tej samej osoby: [['marie patricia hurel', 'marie-patricia hurel'], ['jean charles niel', 'jean-charles niel'], ['jean charles mignon', 'jean-charles mignon']]
  w

In [ ]:
# --- KROKI 2-4: kraje, marki, perfumiarze (tabele slownikowe) + tabela laczaca perfumy z perfumiarzami ---
# identyfikatory perfum w bazie: kolejne liczby 1..N wg kolejnosci wierszy CSV (ta sama kolejnosc co wektory Z i klastry); id Fragrantiki zostaje w osobnej kolumnie
df["perfume_id"] = np.arange(1, N + 1)
df["fragrantica_id"] = index.perfume_id.astype("int64").values

def zapisz(tab, nazwa):
    tab.to_csv(os.path.join(CONFIG["OUTPUT_DIR"], nazwa), index=False, encoding="utf-8")      
    print(f"  zapisano {nazwa}: {len(tab)} wierszy")

# ---------- KRAJE ----------
POPRAWKI_KRAJOW = {"arabia saudi": "Saudi Arabia", "bahrein": "Bahrain"}                      
czysty_kraj = lambda k: POPRAWKI_KRAJOW.get(k.strip().lower(), k.strip())
df["kraj"] = df.Country.map(czysty_kraj)
countries = pd.DataFrame({"name": sorted(df.kraj.unique())})
countries.insert(0, "country_id", np.arange(1, len(countries) + 1))
kraj_id = dict(zip(countries.name, countries.country_id))

# ---------- MARKI  ----------
marki = df.groupby("Brand", sort=True).kraj.agg(["first", "nunique"]).reset_index()
assert (marki["nunique"] == 1).all(), "marka z wiecej niz jednym krajem"
brands = pd.DataFrame({"brand_id": np.arange(1, len(marki) + 1), "name": marki.Brand, "country_id": marki["first"].map(kraj_id)})
marka_id = dict(zip(brands.name, brands.brand_id))
df["brand_id"] = df.Brand.map(marka_id)

# ---------- PERFUMIARZE ----------
POPRAWKI_PERFUMIARZY = {"aure?lien guichard": "aurélien guichard", "myle?ne alran": "mylène alran", "ste?phanie de bruijn": "stéphanie de bruijn",
                        "hüseyin erdo?mu?": "hüseyin erdoğmuş", "gökhan ?im?ek": "gökhan şimşek",
                        "jean françois latty": "jean-françois latty", "jean-francois latty": "jean-françois latty", "cara sylvain": "sylvain cara",
                        "david appel": "david apel", "marrypierre julien": "marypierre julien", "michelle saramitot": "michèle saramito",
                        "olivier glotin": "olivier gillotin", "vicent schaller": "vincent schaller"}
FIRMY = {"firmenich", "givaudan", "iff", "mane", "robertet", "symrise", "takasago", "puig", "quest", "dragoco", "drom", "ungerer & company"}
usun_znaki_sterujace = lambda s: re.sub(r"[\x80-\x9f]", "", s)                               

surowe = collections.Counter(t.strip() for x in df.Perfumers for t in x.split(",") if t.strip())
rodziny = collections.defaultdict(list)
for nazwa in surowe:
    rodziny[re.sub(r"[^a-z]", "", nazwa)].append(nazwa)
kanoniczny = {}                                                                              
for warianty in rodziny.values():
    cel = next((w for w in warianty if "-" in w), max(warianty, key=surowe.get))
    for w in warianty:
        kanoniczny[w] = cel
def czysty_perfumiarz(nazwa):
    nazwa = usun_znaki_sterujace(nazwa.strip())
    nazwa = POPRAWKI_PERFUMIARZY.get(nazwa, nazwa)
    return kanoniczny.get(nazwa, nazwa)

df["perfumiarze_lista"] = df.Perfumers.map(lambda x: list(dict.fromkeys(czysty_perfumiarz(t) for t in x.split(",") if t.strip() and t.strip().lower() != "unknown")))
nazwy_perfumiarzy = sorted({p for l in df.perfumiarze_lista for p in l})
perfumers = pd.DataFrame({"perfumer_id": np.arange(1, len(nazwy_perfumiarzy) + 1), "name": nazwy_perfumiarzy})
perfumers["is_company"] = np.where(perfumers.name.isin(FIRMY), "true", "false")              # tekst 'true'/'false' - PostgreSQL wczyta jako boolean
perfumer_id = dict(zip(perfumers.name, perfumers.perfumer_id))
perfume_perfumers = pd.DataFrame([(pid, perfumer_id[p]) for pid, l in zip(df.perfume_id, df.perfumiarze_lista) for p in l], columns=["perfume_id", "perfumer_id"])

# ---------- zapis ----------
print("zapis plikow:")
zapisz(countries, "countries.csv"); zapisz(brands, "brands.csv"); zapisz(perfumers, "perfumers.csv"); zapisz(perfume_perfumers, "perfume_perfumers.csv")

# ---------- kontrole ----------
kontrole = {
    "kraje: 53 unikalne nazwy, bez spacji na brzegach":                 len(countries) == 53 and countries.name.is_unique and bool((countries.name == countries.name.str.strip()).all()),
    "marki: 1060, unikalne nazwy, kazda ma kraj z tabeli countries":    len(brands) == 1060 and brands.name.is_unique and bool(brands.country_id.isin(countries.country_id).all()),
    "kazda perfuma ma marke z tabeli brands":                           bool(df.brand_id.notna().all() and df.brand_id.isin(brands.brand_id).all()),
    "perfumiarze: 899 unikalnych nazw, bez 'unknown', 12 firm":         len(perfumers) == 899 and perfumers.name.is_unique and "unknown" not in set(perfumers.name) and int((perfumers.is_company == "true").sum()) == 12,
    "perfume_perfumers: pary unikalne, klucze obce poprawne":           (not perfume_perfumers.duplicated().any()) and bool(perfume_perfumers.perfume_id.between(1, N).all()) and bool(perfume_perfumers.perfumer_id.isin(perfumers.perfumer_id).all()),
    "liczba powiazan = liczba wpisow poza 'unknown' (25379 - 12310)":   len(perfume_perfumers) == sum(v for k, v in surowe.items() if k.lower() != "unknown"),
}
print("\nkontrole:")
for k, v in kontrole.items():
    print(f"  {'OK ' if v else 'BLAD'}  {k}")
assert all(kontrole.values())
print(f"\nperfumy z co najmniej jednym znanym perfumiarzem: {int((df.perfumiarze_lista.str.len() > 0).sum())} z {N}; z dwoma: {int((df.perfumiarze_lista.str.len() == 2).sum())}")
print("nazwy perfumiarzy nadal z '?' (do recznej poprawki):", [n for n in perfumers.name if "?" in n])
print("najczesciej wystepujacy perfumiarze:", perfume_perfumers.perfumer_id.map(dict(zip(perfumers.perfumer_id, perfumers.name))).value_counts().head(5).to_dict())
print("kraje marek (top 5):", brands.merge(countries, on="country_id").name_y.value_counts().head(5).to_dict())

zapis plikow:
  zapisano countries.csv: 53 wierszy
  zapisano brands.csv: 1060 wierszy
  zapisano perfumers.csv: 899 wierszy
  zapisano perfume_perfumers.csv: 13069 wierszy

kontrole:
  OK   kraje: 53 unikalne nazwy, bez spacji na brzegach
  OK   marki: 1060, unikalne nazwy, kazda ma kraj z tabeli countries
  OK   kazda perfuma ma marke z tabeli brands
  OK   perfumiarze: 899 unikalnych nazw, bez 'unknown', 12 firm
  OK   perfume_perfumers: pary unikalne, klucze obce poprawne
  OK   liczba powiazan = liczba wpisow poza 'unknown' (25379 - 12310)

perfumy z co najmniej jednym znanym perfumiarzem: 11736 z 24046; z dwoma: 1333
nazwy perfumiarzy nadal z '?' (do recznej poprawki): ['liesma oe (pr?se)']
najczesciej wystepujacy perfumiarze: {'alberto morillas': 340, 'olivier cresp': 255, 'verônica kato': 194, 'nathalie lorson': 172, 'thierry wasser': 149}
kraje marek (top 5): {'France': 275, 'USA': 229, 'Italy': 152, 'UK': 70, 'UAE': 50}


In [ ]:
# --- KROK 5: nuty szczegolowe (notes) i powiazania perfum z nutami w warstwach piramidy (perfume_notes) ---
# Zrodlo: kolumny Top / Middle / Base, nuty rozdzielone przecinkami. Ta sama nuta moze wystapic w kilku warstwach tej samej perfumy (1082 perfumy),
# dlatego kluczem glownym perfume_notes jest trojka (perfume_id, note_id, layer).
WARSTWY = {"Top": "top", "Middle": "middle", "Base": "base"}                             

def rozbij_nuty(tekst):
    return list(dict.fromkeys(t.strip() for t in tekst.split(",") if t.strip()))           

wiersze = []
for kol, warstwa in WARSTWY.items():
    for pid, tekst in zip(df.perfume_id, df[kol]):
        for nuta in rozbij_nuty(tekst):
            wiersze.append((pid, nuta, warstwa))
pn_surowe = pd.DataFrame(wiersze, columns=["perfume_id", "nota", "layer"])

# tabela nut szczegolowych: jedna nazwa = jeden wiersz, identyfikatory wg kolejnosci alfabetycznej
notes = pd.DataFrame({"name": sorted(pn_surowe.nota.unique())})
notes.insert(0, "note_id", np.arange(1, len(notes) + 1))
nuta_id = dict(zip(notes.name, notes.note_id))

perfume_notes = pn_surowe.assign(note_id=pn_surowe.nota.map(nuta_id))[["perfume_id", "note_id", "layer"]]
perfume_notes["_kol"] = perfume_notes.layer.map({"top": 0, "middle": 1, "base": 2})
perfume_notes = perfume_notes.sort_values(["perfume_id", "_kol", "note_id"]).drop(columns="_kol").reset_index(drop=True)
zapisz(perfume_notes, "perfume_notes.csv")                                                 

# --- kontrole ---
liczba_zrodlowa = sum(len(rozbij_nuty(t)) for kol in WARSTWY for t in df[kol])
kontrole = {
    "nuty: 1664 unikalne nazwy":                                        len(notes) == 1664 and notes.name.is_unique,
    "perfume_notes: liczba wierszy = liczba nut w warstwach (238208)":  len(perfume_notes) == liczba_zrodlowa == 238208,
    "klucz glowny (perfume_id, note_id, layer) unikalny":               not perfume_notes.duplicated(["perfume_id", "note_id", "layer"]).any(),
    "klucze obce: perfume_id 1..N, note_id z tabeli notes":             bool(perfume_notes.perfume_id.between(1, N).all()) and bool(perfume_notes.note_id.isin(notes.note_id).all()),
    "tylko warstwy top/middle/base":                                    set(perfume_notes.layer) == {"top", "middle", "base"},
    "kazda perfuma ma nuty w kazdej z trzech warstw":                   bool((perfume_notes.groupby("perfume_id").layer.nunique() == 3).all()) and perfume_notes.perfume_id.nunique() == N,
    "kazda nuta wystepuje w co najmniej jednej perfumie":               bool(notes.note_id.isin(perfume_notes.note_id).all()),
}
print("\nkontrole:")
for k, v in kontrole.items():
    print(f"  {'OK ' if v else 'BLAD'}  {k}")
assert all(kontrole.values())

# --- statystyki do wgladu ---
print("\nwiersze wg warstwy:", perfume_notes.layer.value_counts().to_dict())
print("nut na perfume (srednia/mediana/max):", round(perfume_notes.groupby("perfume_id").size().mean(), 1), "/", int(perfume_notes.groupby("perfume_id").size().median()), "/", int(perfume_notes.groupby("perfume_id").size().max()))
dwuwarstwowe = perfume_notes.groupby(["perfume_id", "note_id"]).size()
print(f"pary (perfuma, nuta) wystepujace w wiecej niz jednej warstwie: {int((dwuwarstwowe > 1).sum())} w {dwuwarstwowe[dwuwarstwowe > 1].reset_index().perfume_id.nunique()} perfumach")
czestosc = perfume_notes.groupby("note_id").perfume_id.nunique().rename("perfum")
print("najczestsze nuty:", notes.merge(czestosc, left_on="note_id", right_index=True).sort_values("perfum", ascending=False).head(6)[["name", "perfum"]].values.tolist())
print(f"nuty w >=100 perfumach: {int((czestosc >= 100).sum())}, w 1 perfumie: {int((czestosc == 1).sum())}, nuta z nawiasem: {[n for n in notes.name if re.search(r'[\\(\\[]', n)]}")

  zapisano perfume_notes.csv: 238208 wierszy

kontrole:
  OK   nuty: 1664 unikalne nazwy
  OK   perfume_notes: liczba wierszy = liczba nut w warstwach (238208)
  OK   klucz glowny (perfume_id, note_id, layer) unikalny
  OK   klucze obce: perfume_id 1..N, note_id z tabeli notes
  OK   tylko warstwy top/middle/base
  OK   kazda perfuma ma nuty w kazdej z trzech warstw
  OK   kazda nuta wystepuje w co najmniej jednej perfumie

wiersze wg warstwy: {'base': 84127, 'middle': 79278, 'top': 74803}
nut na perfume (srednia/mediana/max): 9.9 / 9 / 66
pary (perfuma, nuta) wystepujace w wiecej niz jednej warstwie: 1267 w 1082 perfumach
najczestsze nuty: [['musk', 10942], ['bergamot', 8608], ['sandalwood', 8007], ['amber', 7681], ['jasmine', 7669], ['patchouli', 7204]]
nuty w >=100 perfumach: 271, w 1 perfumie: 345, nuta z nawiasem: ['frosting [glacé]']


In [ ]:
# --- KROK 6: akordy (accords) i ich pozycje w perfumach (perfume_accords) ---
# mainaccord1..mainaccord5 = akordy glowne perfumy od najbardziej wyczuwalnego (1) do najmniej (5)
KOL_AKORDOW = [f"mainaccord{i}" for i in range(1, 6)]
akordy_surowe = pd.concat([df[k] for k in KOL_AKORDOW]).dropna().str.strip()
accords = pd.DataFrame({"name": sorted(akordy_surowe.unique())})
accords.insert(0, "accord_id", np.arange(1, len(accords) + 1))
akord_id = dict(zip(accords.name, accords.accord_id))

wiersze = [(pid, akord_id[a.strip()], poz)
           for pid, wiersz in zip(df.perfume_id, df[KOL_AKORDOW].values)
           for poz, a in enumerate(wiersz, start=1) if isinstance(a, str) and a.strip()]
perfume_accords = pd.DataFrame(wiersze, columns=["perfume_id", "accord_id", "accord_rank"])
print("zapis plikow:")
zapisz(accords, "accords.csv"); zapisz(perfume_accords, "perfume_accords.csv")

ile_na_perfumy = perfume_accords.groupby("perfume_id").accord_rank.agg(["count", "max"])
kontrole = {
    "akordy: 84 unikalne nazwy":                                          len(accords) == 84 and accords.name.is_unique,
    "perfume_accords: 118736 wierszy":                                    len(perfume_accords) == 118736,
    "pozycja akordu tylko 1..5":                                          bool(perfume_accords.accord_rank.between(1, 5).all()),
    "para (perfuma, akord) unikalna - ten sam akord nie wystepuje dwa razy": not perfume_accords.duplicated(["perfume_id", "accord_id"]).any(),
    "para (perfuma, pozycja) unikalna - jedna pozycja = jeden akord":     not perfume_accords.duplicated(["perfume_id", "accord_rank"]).any(),
    "klucze obce: perfume_id 1..N, accord_id z tabeli accords":           bool(perfume_accords.perfume_id.between(1, N).all()) and bool(perfume_accords.accord_id.isin(accords.accord_id).all()),
    "kazda perfuma ma akord na pozycji 1":                                bool((perfume_accords.query("accord_rank == 1").perfume_id.nunique() == N)),
    "pozycje bez dziur (jesli jest pozycja k, sa wszystkie 1..k)":        bool((ile_na_perfumy["count"] == ile_na_perfumy["max"]).all()),
    "kazdy akord wystepuje w co najmniej jednej perfumie":                bool(accords.accord_id.isin(perfume_accords.accord_id).all()),
}
print("\nkontrole:")
for k, v in kontrole.items():
    print(f"  {'OK ' if v else 'BLAD'}  {k}")
assert all(kontrole.values())

print("\nliczba akordow na perfume:", ile_na_perfumy["count"].value_counts().sort_index().to_dict())
nazwa_akordu = dict(zip(accords.accord_id, accords.name))
print("najczestszy akord na pozycji 1:", perfume_accords.query("accord_rank == 1").accord_id.map(nazwa_akordu).value_counts().head(6).to_dict())
rzadkie = perfume_accords.groupby("accord_id").perfume_id.nunique()
print(f"najrzadsze akordy (liczba perfum): {[(nazwa_akordu[i], int(n)) for i, n in rzadkie.sort_values().head(8).items()]}")

zapis plikow:
  zapisano accords.csv: 84 wierszy
  zapisano perfume_accords.csv: 118736 wierszy

kontrole:
  OK   akordy: 84 unikalne nazwy
  OK   perfume_accords: 118736 wierszy
  OK   pozycja akordu tylko 1..5
  OK   para (perfuma, akord) unikalna - ten sam akord nie wystepuje dwa razy
  OK   para (perfuma, pozycja) unikalna - jedna pozycja = jeden akord
  OK   klucze obce: perfume_id 1..N, accord_id z tabeli accords
  OK   kazda perfuma ma akord na pozycji 1
  OK   pozycje bez dziur (jesli jest pozycja k, sa wszystkie 1..k)
  OK   kazdy akord wystepuje w co najmniej jednej perfumie

liczba akordow na perfume: {1: 13, 2: 101, 3: 274, 4: 591, 5: 23067}
najczestszy akord na pozycji 1: {'citrus': 3313, 'woody': 3150, 'fruity': 2278, 'white floral': 1935, 'floral': 1682, 'aromatic': 1665}
najrzadsze akordy (liczba perfum): [('brown scotch tape', 1), ('creamy', 1), ('industrial glue', 1), ('gasoline', 1), ('paper', 1), ('rubber', 1), ('plastic', 2), ('oily', 2)]


In [ ]:
# --- KROK 7: clusters, perfumes, perfume_embeddings ---
# (a) clusters: 7 rodzin + 17 podrodzin z clustering.ipynb + srodek klastra (wektor 512) - backend uzywa srodkow do przypisania nowych perfum
kl = pd.read_csv(os.path.join(CLU_DIR, "clusters.csv"))
srodki = np.load(os.path.join(CLU_DIR, "srodki_klastrow.npy"))
srodki_id = pd.read_csv(os.path.join(CLU_DIR, "srodki_klastrow_id.csv")).cluster_id.tolist()
assert set(srodki_id) == set(kl.cluster_id) and srodki.shape[1] == Z.shape[1], "srodki nie pasuja do clusters.csv"
wektor_tekst = lambda M: ["[" + ",".join(f"{v:.6f}" for v in w) + "]" for w in M]            # format wektora w pgvector: [x1,x2,...,x512]
clusters = kl[["cluster_id", "level", "parent_cluster_id", "name", "description"]].copy()
clusters["parent_cluster_id"] = clusters.parent_cluster_id.astype("Int64")                       
clusters["centroid"] = pd.Series(wektor_tekst(srodki[[srodki_id.index(i) for i in clusters.cluster_id]]), index=clusters.index)
zapisz(clusters, "clusters.csv")

# (b) perfumes: opis perfum + marka + klaster. created_at nadaje baza (DEFAULT now()), wiec nie ma go w pliku
perfumes = pd.DataFrame({
    "perfume_id":          df.perfume_id,
    "fragrantica_id":      df.fragrantica_id,
    "url":                 df.url,
    "name":                df.Perfume,
    "brand_id":            df.brand_id,
    "gender":              df.Gender,                                                            
    "release_year":        df.Year.astype("Int64"),                                              
    "source_rating_value": df["Rating Value"],
    "source_rating_count": df["Rating Count"],
    "cluster_id":          pcl.cluster_id.values,                                                
    "cluster_distance":    pcl.cluster_distance.values,
})
zapisz(perfumes, "perfumes.csv")

# (c) perfume_embeddings: wektor 512 wymiarow (6 miejsc po przecinku wystarcza: pgvector i tak trzyma float4)
t0 = time.time()
perfume_embeddings = pd.DataFrame({"perfume_id": df.perfume_id, "embedding": wektor_tekst(Z)})
zapisz(perfume_embeddings, "perfume_embeddings.csv")
print(f"  (wektory zapisane w {time.time() - t0:.0f} s, plik {os.path.getsize(os.path.join(CONFIG['OUTPUT_DIR'], 'perfume_embeddings.csv')) / 1e6:.0f} MB)")

# --- kontrole ---
def sparsuj(tekst):
    return np.array(tekst.strip("[]").split(","), dtype=np.float32)
roznica_emb = max(float(np.abs(sparsuj(perfume_embeddings.embedding[i]) - Z[i]).max()) for i in np.linspace(0, N - 1, 200).astype(int))
roznica_cen = max(float(np.abs(sparsuj(clusters.centroid[j]) - srodki[srodki_id.index(clusters.cluster_id[j])]).max()) for j in range(len(clusters)))
podrodziny = set(clusters[clusters.level == "subfamily"].cluster_id)
kontrole = {
    "perfumes: 24046 wierszy, unikalne perfume_id, fragrantica_id i url":      len(perfumes) == N and perfumes.perfume_id.is_unique and perfumes.fragrantica_id.is_unique and perfumes.url.is_unique,
    "perfumes: brand_id istnieje w brands":                                   bool(perfumes.brand_id.isin(brands.brand_id).all()),
    "perfumes: plec tylko women/men/unisex":                                  set(perfumes.gender) == {"women", "men", "unisex"},
    "perfumes: brak roku w 2034 perfumach, reszta 1781-2024":                 int(perfumes.release_year.isna().sum()) == 2034 and int(perfumes.release_year.min()) == 1781 and int(perfumes.release_year.max()) == 2024,
    "perfumes: cluster_id wskazuje podrodzine z tabeli clusters":             bool(perfumes.cluster_id.isin(podrodziny).all()),
    "perfumes: cluster_distance w zakresie 0..1.5, bez NaN":                  bool(perfumes.cluster_distance.between(0, 1.5).all()),
    "clusters: 24 wiersze (7 rodzin + 17 podrodzin), podrodziny maja rodzica": len(clusters) == 24 and (clusters.level == "family").sum() == 7 and bool(clusters[clusters.level == "subfamily"].parent_cluster_id.isin(clusters[clusters.level == "family"].cluster_id).all()),
    "clusters: rodziny bez rodzica":                                          bool(clusters[clusters.level == "family"].parent_cluster_id.isna().all()),
    "embeddingi: 24046 wierszy, kazda perfuma ma wektor":                     len(perfume_embeddings) == N and perfume_embeddings.perfume_id.is_unique,
    "embeddingi: tekst odtwarza wektor (maks. roznica < 1e-5, 200 probek)":   roznica_emb < 1e-5,
    "srodki: tekst odtwarza wektor (maks. roznica < 1e-5)":                   roznica_cen < 1e-5,
}
print("\nkontrole:")
for k, v in kontrole.items():
    print(f"  {'OK ' if v else 'BLAD'}  {k}")
assert all(kontrole.values())
print(f"\nmaks. roznica po zaokrągleniu wektora: {roznica_emb:.2e}; perfum w podrodzinach (min/mediana/max): "
      f"{int(perfumes.cluster_id.value_counts().min())}/{int(perfumes.cluster_id.value_counts().median())}/{int(perfumes.cluster_id.value_counts().max())}")
display(perfumes.head(3))

  zapisano clusters.csv: 24 wierszy
  zapisano perfumes.csv: 24046 wierszy
  zapisano perfume_embeddings.csv: 24046 wierszy
  (wektory zapisane w 4 s, plik 117 MB)

kontrole:
  OK   perfumes: 24046 wierszy, unikalne perfume_id, fragrantica_id i url
  OK   perfumes: brand_id istnieje w brands
  OK   perfumes: plec tylko women/men/unisex
  OK   perfumes: brak roku w 2034 perfumach, reszta 1781-2024
  OK   perfumes: cluster_id wskazuje podrodzine z tabeli clusters
  OK   perfumes: cluster_distance w zakresie 0..1.5, bez NaN
  OK   clusters: 24 wiersze (7 rodzin + 17 podrodzin), podrodziny maja rodzica
  OK   clusters: rodziny bez rodzica
  OK   embeddingi: 24046 wierszy, kazda perfuma ma wektor
  OK   embeddingi: tekst odtwarza wektor (maks. roznica < 1e-5, 200 probek)
  OK   srodki: tekst odtwarza wektor (maks. roznica < 1e-5)

maks. roznica po zaokrągleniu wektora: 5.07e-07; perfum w podrodzinach (min/mediana/max): 621/1369/2370


,perfume_id,fragrantica_id,url,name,brand_id,gender,release_year,source_rating_value,source_rating_count,cluster_id,cluster_distance
0,1,93845,https://www.fragrantica.com/perfume/we-pink/wa...,WAFFLE,1037,women,2023,2.41,27,6,0.33656
1,2,81650,https://www.fragrantica.com/perfume/vivamor-pa...,ROUGE IMPERIALE,1036,unisex,2023,2.62,26,13,0.30334
2,3,45795,https://www.fragrantica.com/perfume/brocard/ch...,CHERRY LADY,210,women,<NA>,2.64,39,17,0.28558


In [ ]:
import hashlib

# --- KROK 8: notes.csv i general_notes.csv + kontrola calej paczki odczytanej z plikow ---
general_notes = pd.DataFrame({"general_note_id": pd.Series(dtype="int64"), "name": pd.Series(dtype="object")})
zapisz(general_notes, "general_notes.csv")
zapisz(notes.assign(general_note_id=pd.array([pd.NA] * len(notes), dtype="Int64"))[["note_id", "name", "general_note_id"]], "notes.csv")

OCZEKIWANE = {
    "countries":          (["country_id", "name"], 53),
    "brands":             (["brand_id", "name", "country_id"], 1060),
    "perfumers":          (["perfumer_id", "name", "is_company"], 899),
    "general_notes":      (["general_note_id", "name"], 0),
    "notes":              (["note_id", "name", "general_note_id"], 1664),
    "accords":            (["accord_id", "name"], 84),
    "clusters":           (["cluster_id", "level", "parent_cluster_id", "name", "description", "centroid"], 24),
    "perfumes":           (["perfume_id", "fragrantica_id", "url", "name", "brand_id", "gender", "release_year", "source_rating_value", "source_rating_count", "cluster_id", "cluster_distance"], N),
    "perfume_embeddings": (["perfume_id", "embedding"], N),
    "perfume_perfumers":  (["perfume_id", "perfumer_id"], 13069),
    "perfume_notes":      (["perfume_id", "note_id", "layer"], 238208),
    "perfume_accords":    (["perfume_id", "accord_id", "accord_rank"], 118736),
}
T = {nazwa: pd.read_csv(os.path.join(CONFIG["OUTPUT_DIR"], f"{nazwa}.csv"), keep_default_na=False, na_values=[""], dtype_backend="numpy_nullable") for nazwa in OCZEKIWANE}

kontrole = {}
for nazwa, (kolumny, wiersze) in OCZEKIWANE.items():
    kontrole[f"{nazwa}: kolumny i liczba wierszy ({wiersze})"] = list(T[nazwa].columns) == kolumny and len(T[nazwa]) == wiersze
klucze = {"countries": "country_id", "brands": "brand_id", "perfumers": "perfumer_id", "notes": "note_id", "accords": "accord_id", "clusters": "cluster_id",
          "perfumes": "perfume_id", "perfume_embeddings": "perfume_id"}
kontrole["klucze glowne tabel slownikowych i perfumes: unikalne i niepuste"] = all(T[t][k].is_unique and T[t][k].notna().all() for t, k in klucze.items())
kontrole["klucze glowne tabel laczacych: pary unikalne"] = (not T["perfume_perfumers"].duplicated().any()) and (not T["perfume_notes"].duplicated().any()) \
    and (not T["perfume_accords"].duplicated(["perfume_id", "accord_id"]).any()) and (not T["perfume_accords"].duplicated(["perfume_id", "accord_rank"]).any())
fk = {  
    ("brands", "country_id"): ("countries", "country_id"), ("perfumes", "brand_id"): ("brands", "brand_id"),
    ("perfumes", "cluster_id"): ("clusters", "cluster_id"), ("perfume_embeddings", "perfume_id"): ("perfumes", "perfume_id"),
    ("perfume_perfumers", "perfume_id"): ("perfumes", "perfume_id"), ("perfume_perfumers", "perfumer_id"): ("perfumers", "perfumer_id"),
    ("perfume_notes", "perfume_id"): ("perfumes", "perfume_id"), ("perfume_notes", "note_id"): ("notes", "note_id"),
    ("perfume_accords", "perfume_id"): ("perfumes", "perfume_id"), ("perfume_accords", "accord_id"): ("accords", "accord_id"),
    ("clusters", "parent_cluster_id"): ("clusters", "cluster_id"), ("notes", "general_note_id"): ("general_notes", "general_note_id"),
}
for (t, k), (t2, k2) in fk.items():
    wartosci = T[t][k].dropna()
    kontrole[f"klucz obcy {t}.{k} -> {t2}.{k2}"] = bool(wartosci.isin(T[t2][k2]).all())
podr = set(T["clusters"].query("level == 'subfamily'").cluster_id)
kontrole["perfumes.cluster_id wskazuje tylko podrodziny"] = bool(T["perfumes"].cluster_id.isin(podr).all())
kontrole["kazda perfuma ma embedding, akord na pozycji 1 i nuty"] = (set(T["perfume_embeddings"].perfume_id) == set(T["perfumes"].perfume_id)
                                                                      and T["perfume_accords"].query("accord_rank == 1").perfume_id.nunique() == N and T["perfume_notes"].perfume_id.nunique() == N)
kontrole["wektory: dokladnie 512 liczb w kazdym tekscie (embeddingi i srodki)"] = bool((T["perfume_embeddings"].embedding.str.count(",") == 511).all() and (T["clusters"].centroid.str.count(",") == 511).all()
                                                                                        and T["perfume_embeddings"].embedding.str.startswith("[").all() and T["perfume_embeddings"].embedding.str.endswith("]").all())
kontrole["boolean is_company: tylko true/false (12 firm)"] = set(T["perfumers"].is_company.astype(str).str.lower()) <= {"true", "false"} and int(T["perfumers"].is_company.astype(bool).sum()) == 12   # pandas sam rozpoznaje true/false jako typ logiczny
print("kontrola calej paczki odczytanej z plikow:")
for k, v in kontrole.items():
    print(f"  {'OK ' if v else 'BLAD'}  {k}")
assert all(bool(v) for v in kontrole.values()), "paczka niespojna - nie ladujemy do bazy"

# manifest paczki: rozmiary i odciski plikow (po zaladowaniu do bazy wiadomo, z jakiej wersji plikow powstala)
def sha_pliku(plik):
    h = hashlib.sha256()
    with open(plik, "rb") as f:
        for kawalek in iter(lambda: f.read(1 << 20), b""):
            h.update(kawalek)
    return h.hexdigest()[:16]
manifest_bazy = {nazwa: {"wiersze": int(len(T[nazwa])), "bajtow": os.path.getsize(os.path.join(CONFIG["OUTPUT_DIR"], f"{nazwa}.csv")),
                         "sha256_16": sha_pliku(os.path.join(CONFIG["OUTPUT_DIR"], f"{nazwa}.csv"))} for nazwa in OCZEKIWANE}
with open(os.path.join(CONFIG["OUTPUT_DIR"], "baza_import_manifest.json"), "w", encoding="utf-8") as f:
    json.dump({"data": time.strftime("%Y-%m-%d"), "pliki": manifest_bazy}, f, ensure_ascii=False, indent=1)
display(pd.DataFrame(manifest_bazy).T)

  zapisano general_notes.csv: 0 wierszy
  zapisano notes.csv: 1664 wierszy
kontrola calej paczki odczytanej z plikow:
  OK   countries: kolumny i liczba wierszy (53)
  OK   brands: kolumny i liczba wierszy (1060)
  OK   perfumers: kolumny i liczba wierszy (899)
  OK   general_notes: kolumny i liczba wierszy (0)
  OK   notes: kolumny i liczba wierszy (1664)
  OK   accords: kolumny i liczba wierszy (84)
  OK   clusters: kolumny i liczba wierszy (24)
  OK   perfumes: kolumny i liczba wierszy (24046)
  OK   perfume_embeddings: kolumny i liczba wierszy (24046)
  OK   perfume_perfumers: kolumny i liczba wierszy (13069)
  OK   perfume_notes: kolumny i liczba wierszy (238208)
  OK   perfume_accords: kolumny i liczba wierszy (118736)
  OK   klucze glowne tabel slownikowych i perfumes: unikalne i niepuste
  OK   klucze glowne tabel laczacych: pary unikalne
  OK   klucz obcy brands.country_id -> countries.country_id
  OK   klucz obcy perfumes.brand_id -> brands.brand_id
  OK   klucz obcy perfumes

,wiersze,bajtow,sha256_16
countries,53,634,ec41fa20bec3935d
brands,1060,21966,3d6c7feccfad284f
perfumers,899,23994,6c9f8e6f97473a3b
general_notes,0,22,cc2fac1b006999f9
notes,1664,30232,514ca3fe1ee7556d
accords,84,999,330514c87e7c5343
clusters,24,119934,1e88c389f88f45c1
perfumes,24046,3315272,183294f0cf105265
perfume_embeddings,24046,117276106,9ca60b210431df13
perfume_perfumers,13069,135890,5476ee6b571789b8


In [1]:
%pip install -q "psycopg[binary]" python-dotenv

import os
from dotenv import load_dotenv
import psycopg

load_dotenv(os.path.join("..", ".env"))                    # plik .env w glownym folderze repozytorium (notatnik lezy w notatniki/)
POLACZENIE = dict(host=os.environ["DB_HOST"], port=int(os.environ["DB_PORT"]), dbname=os.environ["DB_NAME"],
                  user=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"], sslmode="require")   # sslmode: polaczenie szyfrowane

with psycopg.connect(**POLACZENIE, connect_timeout=15) as conn:
    with conn.cursor() as cur:
        cur.execute("select version()")
        print("serwer:", cur.fetchone()[0])
        cur.execute("select current_user, current_database()")
        print("uzytkownik, baza:", cur.fetchone())
        cur.execute("select name, default_version, installed_version from pg_available_extensions where name in ('vector', 'pg_trgm') order by name")
        print("rozszerzenia (nazwa, wersja dostepna, zainstalowana):", cur.fetchall())


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
serwer: PostgreSQL 17.11 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit
uzytkownik, baza: ('postgres', 'postgres')
rozszerzenia (nazwa, wersja dostepna, zainstalowana): [('pg_trgm', '1.6', None), ('vector', '0.8.2', None)]


In [ ]:
import pathlib, re

# --- KROK 6: tworzenie schematu (tabele, typy, ograniczenia) na bazie w Supabase ---
RESET_PRZED_TWORZENIEM = True    # True: najpierw usuwa nasze tabele i typy (pozwala uruchomic komorke ponownie). Gdy w bazie beda dane uzytkownikow, zmienic na False
TABELE = ["perfume_accords", "perfume_notes", "perfume_perfumers", "perfume_embeddings", "perfumes", "clusters", "notes", "general_notes", "accords", "perfumers", "brands", "countries"]
TYPY = ["gender_t", "layer_t", "cluster_level_t"]

SCIEZKA_SQL = pathlib.Path("..") / "baza" / "01_schema.sql"
assert SCIEZKA_SQL.exists(), f"brak pliku {SCIEZKA_SQL.resolve()} - skopiuj skrypty SQL do folderu baza/ w glownym folderze repozytorium"
sql = SCIEZKA_SQL.read_text(encoding="utf-8")
sql = "\n".join(l for l in sql.splitlines() if not l.lstrip().startswith("\\"))                      
sql = re.sub(r"CREATE EXTENSION IF NOT EXISTS (\w+);", r"CREATE EXTENSION IF NOT EXISTS \1 WITH SCHEMA extensions;", sql)  
with psycopg.connect(**POLACZENIE, connect_timeout=15) as conn:                                      
    conn.execute("SET search_path TO public, extensions")                                              
    if RESET_PRZED_TWORZENIEM:
        for t in TABELE:
            conn.execute(f"DROP TABLE IF EXISTS public.{t} CASCADE")
        for t in TYPY:
            conn.execute(f"DROP TYPE IF EXISTS public.{t} CASCADE")
    conn.execute(sql)
    tabele = [r[0] for r in conn.execute("select table_name from information_schema.tables where table_schema = 'public' order by 1")]
    typ_wektora = conn.execute("select udt_name from information_schema.columns where table_name = 'perfume_embeddings' and column_name = 'embedding'").fetchone()
    rozszerzenia = conn.execute("select e.extname, e.extversion, n.nspname from pg_extension e join pg_namespace n on n.oid = e.extnamespace where e.extname in ('vector', 'pg_trgm') order by 1").fetchall()
print(f"tabele w schemacie public ({len(tabele)}):", tabele)
print("typ kolumny perfume_embeddings.embedding:", typ_wektora)
print("rozszerzenia (nazwa, wersja, schemat):", rozszerzenia)

tabele w schemacie public (12): ['accords', 'brands', 'clusters', 'countries', 'general_notes', 'notes', 'perfume_accords', 'perfume_embeddings', 'perfume_notes', 'perfume_perfumers', 'perfumers', 'perfumes']
typ kolumny perfume_embeddings.embedding: ('vector',)
rozszerzenia (nazwa, wersja, schemat): [('pg_trgm', '1.6', 'extensions'), ('vector', '0.8.2', 'extensions')]


In [ ]:
import time, pathlib

# --- KROK 7: ladowanie plikow CSV z baza_import do bazy  ---
FOLDER_CSV = pathlib.Path("baza_import")
WYCZYSC_PRZED_LADOWANIEM = True      
KOLEJNOSC = [                        # (tabela, kolumny w kolejnosci z pliku CSV)
    ("countries",          "country_id, name"),
    ("brands",             "brand_id, name, country_id"),
    ("perfumers",          "perfumer_id, name, is_company"),
    ("accords",            "accord_id, name"),
    ("general_notes",      "general_note_id, name"),
    ("notes",              "note_id, name, general_note_id"),
    ("clusters",           "cluster_id, level, parent_cluster_id, name, description, centroid"),
    ("perfumes",           "perfume_id, fragrantica_id, url, name, brand_id, gender, release_year, source_rating_value, source_rating_count, cluster_id, cluster_distance"),
    ("perfume_embeddings", "perfume_id, embedding"),
    ("perfume_perfumers",  "perfume_id, perfumer_id"),
    ("perfume_notes",      "perfume_id, note_id, layer"),
    ("perfume_accords",    "perfume_id, accord_id, accord_rank"),
]
OCZEKIWANE_WIERSZE = {"countries": 53, "brands": 1060, "perfumers": 899, "accords": 84, "general_notes": 0, "notes": 1664, "clusters": 24, "perfumes": 24046,
                      "perfume_embeddings": 24046, "perfume_perfumers": 13069, "perfume_notes": 238208, "perfume_accords": 118736}
KOLUMNY_ID = [("countries", "country_id"), ("brands", "brand_id"), ("perfumers", "perfumer_id"), ("accords", "accord_id"),
              ("general_notes", "general_note_id"), ("notes", "note_id"), ("perfumes", "perfume_id")]       # kolumny z licznikiem automatycznym (IDENTITY)

t_start = time.time()
with psycopg.connect(**POLACZENIE, connect_timeout=15) as conn:            # jedna transakcja: blad = wycofanie wszystkiego
    conn.execute("SET search_path TO public, extensions")
    if WYCZYSC_PRZED_LADOWANIEM:
        # jedno polecenie ze wszystkimi tabelami; gdyby jakas inna tabela (np. uzytkownikow) wskazywala na nasze, baza zglosi blad zamiast skasowac cudze dane
        conn.execute("TRUNCATE " + ", ".join(t for t, _ in KOLEJNOSC))
    for tabela, kolumny in KOLEJNOSC:
        plik = FOLDER_CSV / f"{tabela}.csv"
        t0 = time.time()
        with conn.cursor() as cur:
            with cur.copy(f"COPY {tabela} ({kolumny}) FROM STDIN WITH (FORMAT csv, HEADER true)") as copy:      # strumien pliku do bazy (po stronie klienta)
                with open(plik, "rb") as f:
                    while kawalek := f.read(1 << 20):
                        copy.write(kawalek)
        print(f"  {tabela:20s} wczytano w {time.time() - t0:5.1f} s")
    # kolumny IDENTITY dostaly jawne id z plikow - licznik trzeba ustawic za najwiekszym id, inaczej pierwszy INSERT bez id zderzylby sie z istniejacym
    for tabela, kol in KOLUMNY_ID:
        conn.execute(f"SELECT setval(pg_get_serial_sequence('{tabela}', '{kol}'), COALESCE((SELECT max({kol}) FROM {tabela}), 1), (SELECT count(*) > 0 FROM {tabela}))")
    # kontrola: liczby wierszy w bazie
    print("\nkontrola liczby wierszy (baza / oczekiwane):")
    wszystko_ok = True
    for tabela, _ in KOLEJNOSC:
        w_bazie = conn.execute(f"SELECT count(*) FROM {tabela}").fetchone()[0]
        ok = w_bazie == OCZEKIWANE_WIERSZE[tabela]
        wszystko_ok &= ok
        print(f"  {'OK ' if ok else 'BLAD'}  {tabela:20s} {w_bazie:>8} / {OCZEKIWANE_WIERSZE[tabela]}")
    wymiary, norma = conn.execute("SELECT vector_dims(embedding), round(vector_norm(embedding)::numeric, 4) FROM perfume_embeddings WHERE perfume_id = 1").fetchone()
    print(f"\nwektor perfumy nr 1: {wymiary} wymiarow, dlugosc {norma} (oczekiwane: 512 i 1.0)")
    assert wszystko_ok and wymiary == 512, "niezgodnosc po zaladowaniu - transakcja zostanie wycofana"
print(f"\nzatwierdzono; calosc {time.time() - t_start:.0f} s")

  countries            wczytano w   0.1 s
  brands               wczytano w   0.1 s
  perfumers            wczytano w   0.1 s
  accords              wczytano w   0.1 s
  general_notes        wczytano w   0.1 s
  notes                wczytano w   0.1 s
  clusters             wczytano w   0.1 s
  perfumes             wczytano w   0.9 s
  perfume_embeddings   wczytano w  10.8 s
  perfume_perfumers    wczytano w   0.5 s
  perfume_notes        wczytano w   5.9 s
  perfume_accords      wczytano w   3.2 s

kontrola liczby wierszy (baza / oczekiwane):
  OK   countries                  53 / 53
  OK   brands                   1060 / 1060
  OK   perfumers                 899 / 899
  OK   accords                    84 / 84
  OK   general_notes               0 / 0
  OK   notes                    1664 / 1664
  OK   clusters                   24 / 24
  OK   perfumes                24046 / 24046
  OK   perfume_embeddings      24046 / 24046
  OK   perfume_perfumers       13069 / 13069
  OK   perfume_no

In [ ]:
import pathlib, re, time
import pandas as pd

# --- KROK 8: indeksy i kontrole z pliku 03_indexes_and_checks.sql ---
tekst = (pathlib.Path("..") / "baza" / "03_indexes_and_checks.sql").read_text(encoding="utf-8")
czesci = re.split(r"^\\echo '(.*)'\s*$", tekst, flags=re.M)                 # [poczatek z indeksami, naglowek 1, SQL 1, naglowek 2, SQL 2, ...]
usun_psql = lambda s: "\n".join(l for l in s.splitlines() if not l.lstrip().startswith("\\"))   

t0 = time.time()
with psycopg.connect(**POLACZENIE, connect_timeout=15) as conn:
    conn.execute("SET search_path TO public, extensions")
    conn.execute("SET statement_timeout = 0")                                
    conn.execute("SET maintenance_work_mem = '128MB'")                       # wiecej pamieci na budowe indeksu HNSW (24 tys. wektorow x 512 wymiarow)
    conn.execute(usun_psql(czesci[0]))                                       # CREATE INDEX ... oraz ANALYZE
    print(f"indeksy utworzone w {time.time() - t0:.0f} s\n")
    for naglowek, sql in zip(czesci[1::2], czesci[2::2]):
        print(naglowek)
        kursor = conn.execute(usun_psql(sql))
        display(pd.DataFrame(kursor.fetchall(), columns=[k.name for k in kursor.description]))
    rozmiar = conn.execute("select pg_size_pretty(pg_database_size(current_database()))").fetchone()[0]
    najwieksze = conn.execute("""select c.relname, c.relkind, pg_size_pretty(pg_total_relation_size(c.oid)) from pg_class c join pg_namespace n on n.oid = c.relnamespace
                                 where n.nspname = 'public' and c.relkind in ('r', 'i') order by pg_total_relation_size(c.oid) desc limit 6""").fetchall()
print(f"rozmiar calej bazy: {rozmiar} (limit planu darmowego: 500 MB; w tym schematy systemowe Supabase)")
print("najwieksze obiekty (nazwa, r=tabela / i=indeks, rozmiar):", najwieksze)

indeksy utworzone w 26 s

--- liczby wierszy (oczekiwane: 53 / 1060 / 899 / 84 / 0 / 1664 / 24 / 24046 / 24046 / 13069 / 238208 / 118736) ---


,countries,brands,perfumers,accords,general_notes,notes,clusters,perfumes,embeddings,perf_perfumers,perf_notes,perf_accords
0,53,1060,899,84,0,1664,24,24046,24046,13069,238208,118736


--- perfumes.cluster_id powinno wskazywac tylko podrodziny (oczekiwane: 0) ---


,perfumy_w_rodzinie_zamiast_podrodziny
0,0


--- liczba perfum w rodzinach (oczekiwane: 6101 / 4525 / 3733 / 3627 / 2744 / 2695 / 621) ---


,cluster_id,name,perfum
0,101,Aromatyczno-świeże,6101
1,102,Ciepłe orientalne,4525
2,104,Owocowo-słodkie,3733
3,103,Białe kwiaty i cytrus,3627
4,105,Róża i świeże kwiaty,2744
5,106,Pudrowo-waniliowe,2695
6,107,Zielono-wodne,621


--- test wektorow: 5 perfum najbardziej podobnych do SAUVAGE (Dior), inna marka (oczekiwane: perfumy meskie, aromatyczne/drzewne) ---


,name,brand,cosinus
0,RAVE FOR MEN,SAPIL,0.972
1,SAUVEGY,IN THE BOX,0.972
2,I AM THE KING,ARD AL ZAAFARAN,0.944
3,KITINOS,THERA COSMETICOS,0.935
4,VODKA WILD,PARIS ELYSEES,0.886


--- test wyszukiwarki po nazwie (literowka): "coco mademoisel" ---


,name,brand,podobienstwo
0,COCO MADEMOISELLE,CHANEL,0.79
1,COCO MADEMOISELLE L EAU,CHANEL,0.60
2,COCO MADEMOISELLE PARFUM,CHANEL,0.58
3,COCO MADEMOISELLE INTENSE,CHANEL,0.56
4,MADEMOISELLE,AZZARO,0.53


rozmiar calej bazy: 182 MB (limit planu darmowego: 500 MB; w tym schematy systemowe Supabase)
najwieksze obiekty (nazwa, r=tabela / i=indeks, rozmiar): [('perfume_embeddings', 'r', '128 MB'), ('perfume_embeddings_hnsw_idx', 'i', '63 MB'), ('perfume_notes', 'r', '19 MB'), ('perfume_accords', 'r', '11 MB'), ('perfumes', 'r', '11 MB'), ('perfume_notes_pkey', 'i', '7352 kB')]
